# Deep Learning Model Training - UCI HAR Dataset
## Centralized Deep Learning Approach

This notebook trains a centralized deep learning model for Human Activity Recognition.

**Target Accuracy:** >85%

**Dataset:** UCI HAR Dataset (561 features, 6 activity classes)


## 1. Setup and Installation


In [ ]:
# Install required packages
%pip install numpy pandas matplotlib seaborn scikit-learn tensorflow joblib -q


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
import os
import json
import joblib
from datetime import datetime

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")


## 2. Mount Google Drive and Upload Dataset

Upload your UCI_HAR_Dataset folder to Google Drive, then mount it here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Set the path to your dataset
# Modify this path according to where you uploaded the dataset in Google Drive
DATASET_PATH = '/content/drive/MyDrive/UCI_HAR_Dataset'

print(f"Dataset path: {DATASET_PATH}")
print(f"Dataset exists: {os.path.exists(DATASET_PATH)}")


## 3. Load and Prepare Data


In [ ]:
def load_uci_har_dataset(dataset_path):
    """
    Load UCI HAR Dataset from the specified path.
    
    Returns:
        X_train, y_train, X_test, y_test, feature_names, activity_labels
    """
    # Load training data
    X_train = np.loadtxt(os.path.join(dataset_path, 'train', 'X_train.txt'))
    y_train = np.loadtxt(os.path.join(dataset_path, 'train', 'y_train.txt'))
    
    # Load test data
    X_test = np.loadtxt(os.path.join(dataset_path, 'test', 'X_test.txt'))
    y_test = np.loadtxt(os.path.join(dataset_path, 'test', 'y_test.txt'))
    
    # Load feature names
    with open(os.path.join(dataset_path, 'features.txt'), 'r') as f:
        feature_names = [line.strip().split()[1] for line in f.readlines()]
    
    # Load activity labels
    with open(os.path.join(dataset_path, 'activity_labels.txt'), 'r') as f:
        activity_labels = [line.strip().split()[1] for line in f.readlines()]
    
    # Convert labels to zero-indexed (original is 1-6, convert to 0-5)
    y_train = y_train - 1
    y_test = y_test - 1
    
    return X_train, y_train, X_test, y_test, feature_names, activity_labels

# Load the dataset
X_train, y_train, X_test, y_test, feature_names, activity_labels = load_uci_har_dataset(DATASET_PATH)

print("Dataset loaded successfully!")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Number of classes: {len(activity_labels)}")
print(f"Activity labels: {activity_labels}")


In [ ]:
# Data exploration
print("\n=== Data Statistics ===")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nX_train min/max: {X_train.min():.4f} / {X_train.max():.4f}")
print(f"X_train mean/std: {X_train.mean():.4f} / {X_train.std():.4f}")

# Class distribution
print("\n=== Class Distribution ===")
unique_train, counts_train = np.unique(y_train, return_counts=True)
unique_test, counts_test = np.unique(y_test, return_counts=True)

print("\nTraining set:")
for label_idx, count in zip(unique_train, counts_train):
    print(f"  {activity_labels[int(label_idx)]}: {count} samples ({count/len(y_train)*100:.1f}%)")

print("\nTest set:")
for label_idx, count in zip(unique_test, counts_test):
    print(f"  {activity_labels[int(label_idx)]}: {count} samples ({count/len(y_test)*100:.1f}%)")


In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Training set distribution
axes[0].bar([activity_labels[int(i)] for i in unique_train], counts_train, color='skyblue')
axes[0].set_title('Training Set - Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Activity')
axes[0].set_ylabel('Number of Samples')
axes[0].tick_params(axis='x', rotation=45)

# Test set distribution
axes[1].bar([activity_labels[int(i)] for i in unique_test], counts_test, color='lightcoral')
axes[1].set_title('Test Set - Class Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Activity')
axes[1].set_ylabel('Number of Samples')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 4. Data Preprocessing


In [ ]:
# Standardize features for better neural network performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data standardized successfully!")
print(f"X_train_scaled mean: {X_train_scaled.mean():.6f}")
print(f"X_train_scaled std: {X_train_scaled.std():.6f}")

# Convert labels to categorical (one-hot encoding)
num_classes = len(activity_labels)
y_train_categorical = to_categorical(y_train, num_classes)
y_test_categorical = to_categorical(y_test, num_classes)

print(f"\nLabels converted to categorical format")
print(f"y_train_categorical shape: {y_train_categorical.shape}")
print(f"y_test_categorical shape: {y_test_categorical.shape}")


## 5. Build Deep Learning Model

We'll create a deep neural network with multiple hidden layers, dropout for regularization, and batch normalization.


In [ ]:
def create_deep_learning_model(input_shape, num_classes):
    """
    Create a deep neural network model for activity recognition.
    
    Args:
        input_shape: Shape of input features (561,)
        num_classes: Number of output classes (6)
    
    Returns:
        Compiled Keras model
    """
    model = models.Sequential([
        # Input layer
        layers.Input(shape=input_shape),
        
        # First hidden layer
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        # Second hidden layer
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        
        # Third hidden layer
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Fourth hidden layer
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        # Output layer
        layers.Dense(num_classes, activation='softmax')
    ])
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Create the model
input_shape = (X_train_scaled.shape[1],)
dl_model = create_deep_learning_model(input_shape, num_classes)

# Display model architecture
dl_model.summary()


## 6. Train the Model


In [ ]:
# Define callbacks
callbacks = [
    # Early stopping to prevent overfitting
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate when validation loss plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-7,
        verbose=1
    ),
    
    # Save best model
    ModelCheckpoint(
        'best_dl_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

print("Callbacks configured successfully!")


In [ ]:
# Train the model
print("Starting training...\n")

history = dl_model.fit(
    X_train_scaled,
    y_train_categorical,
    batch_size=32,
    epochs=100,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining completed!")


## 7. Visualize Training History


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dl_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

# Print best results
best_val_acc = max(history.history['val_accuracy'])
best_val_acc_epoch = history.history['val_accuracy'].index(best_val_acc) + 1
print(f"\nBest Validation Accuracy: {best_val_acc*100:.2f}% (Epoch {best_val_acc_epoch})")


## 8. Evaluate on Test Set


In [ ]:
# Evaluate on test set
print("Evaluating on test set...\n")

test_loss, test_accuracy = dl_model.evaluate(X_test_scaled, y_test_categorical, verbose=0)

print(f"\n{'='*50}")
print(f"DEEP LEARNING MODEL - TEST RESULTS")
print(f"{'='*50}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"{'='*50}")

# Check if target accuracy is met
target_accuracy = 0.85
if test_accuracy >= target_accuracy:
    print(f"\n✓ Target accuracy of {target_accuracy*100}% ACHIEVED!")
else:
    print(f"\n✗ Target accuracy of {target_accuracy*100}% NOT achieved.")
    print(f"  Difference: {(target_accuracy - test_accuracy)*100:.2f}%")


In [ ]:
# Make predictions
y_pred_probs = dl_model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

# Print detailed classification report
print("\n=== Classification Report ===")
print(classification_report(
    y_test,
    y_pred,
    target_names=activity_labels,
    digits=4
))


## 9. Confusion Matrix Visualization


In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=activity_labels, yticklabels=activity_labels,
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix (Raw Counts)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Normalized confusion matrix
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=activity_labels, yticklabels=activity_labels,
            ax=axes[1], cbar_kws={'label': 'Proportion'})
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.savefig('dl_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()


## 10. Per-Class Performance Analysis


In [ ]:
# Calculate per-class accuracy
class_accuracies = cm.diagonal() / cm.sum(axis=1)

# Create DataFrame for better visualization
performance_df = pd.DataFrame({
    'Activity': activity_labels,
    'Accuracy': class_accuracies * 100,
    'Samples': cm.sum(axis=1)
})
performance_df = performance_df.sort_values('Accuracy', ascending=False)

print("\n=== Per-Class Performance ===")
print(performance_df.to_string(index=False))

# Visualize per-class accuracy
plt.figure(figsize=(12, 6))
bars = plt.bar(performance_df['Activity'], performance_df['Accuracy'], 
               color=plt.cm.viridis(performance_df['Accuracy']/100))
plt.axhline(y=test_accuracy*100, color='r', linestyle='--', linewidth=2, label=f'Overall Accuracy: {test_accuracy*100:.2f}%')
plt.title('Per-Class Accuracy - Deep Learning Model', fontsize=14, fontweight='bold')
plt.xlabel('Activity')
plt.ylabel('Accuracy (%)')
plt.ylim([0, 105])
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('dl_per_class_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()


## 11. Save the Model


In [ ]:
# Save the final model in multiple formats

# 1. Save as H5 format
dl_model.save('deep_learning_model.h5')
print("Model saved as 'deep_learning_model.h5'")

# 2. Save as SavedModel format (recommended for TensorFlow Serving)
dl_model.save('deep_learning_model_savedmodel', save_format='tf')
print("Model saved as 'deep_learning_model_savedmodel'")

# 3. Save as TFLite format (for mobile deployment)
converter = tf.lite.TFLiteConverter.from_keras_model(dl_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('deep_learning_model.tflite', 'wb') as f:
    f.write(tflite_model)
print("Model saved as 'deep_learning_model.tflite'")

# 4. Save the scaler for preprocessing
joblib.dump(scaler, 'scaler_dl.pkl')
print("Scaler saved as 'scaler_dl.pkl'")

# Get model file sizes
h5_size = os.path.getsize('deep_learning_model.h5') / 1024  # KB
tflite_size = os.path.getsize('deep_learning_model.tflite') / 1024  # KB

print(f"\nModel Sizes:")
print(f"  H5 format: {h5_size:.2f} KB")
print(f"  TFLite format: {tflite_size:.2f} KB")


## 12. Model Information Summary


In [ ]:
# Create model information dictionary
model_info = {
    'model_type': 'Deep Learning (Centralized)',
    'model_name': 'UCI_HAR_DL_Model',
    'framework': 'TensorFlow/Keras',
    'date_trained': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': 'UCI HAR Dataset',
    'input_shape': list(input_shape),
    'num_classes': num_classes,
    'class_labels': activity_labels,
    'training_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0]),
    'num_features': int(X_train.shape[1]),
    'model_architecture': {
        'total_layers': len(dl_model.layers),
        'total_parameters': int(dl_model.count_params()),
        'trainable_parameters': int(sum([tf.size(w).numpy() for w in dl_model.trainable_weights]))
    },
    'training_config': {
        'batch_size': 32,
        'epochs_trained': len(history.history['accuracy']),
        'optimizer': 'Adam',
        'initial_learning_rate': 0.001,
        'loss_function': 'categorical_crossentropy'
    },
    'performance_metrics': {
        'test_accuracy': float(test_accuracy),
        'test_loss': float(test_loss),
        'best_val_accuracy': float(best_val_acc),
        'target_accuracy_met': bool(test_accuracy >= 0.85)
    },
    'model_sizes': {
        'h5_kb': float(h5_size),
        'tflite_kb': float(tflite_size)
    }
}

# Save model info as JSON
with open('deep_learning_model_info.json', 'w') as f:
    json.dump(model_info, f, indent=4)

print("\n=== MODEL INFORMATION ===")
print(json.dumps(model_info, indent=2))
print("\nModel information saved as 'deep_learning_model_info.json'")


## 13. Download Files to Local Machine


In [ ]:
# Download all important files
from google.colab import files

files_to_download = [
    'deep_learning_model.h5',
    'deep_learning_model.tflite',
    'deep_learning_model_info.json',
    'scaler_dl.pkl',
    'dl_training_history.png',
    'dl_confusion_matrix.png',
    'dl_per_class_accuracy.png'
]

print("Downloading files...")
for file in files_to_download:
    if os.path.exists(file):
        files.download(file)
        print(f"  ✓ {file}")
    else:
        print(f"  ✗ {file} not found")

print("\nDownload complete!")


## 14. Test Model Inference


In [ ]:
# Test inference on a few random samples
num_samples = 5
random_indices = np.random.choice(len(X_test_scaled), num_samples, replace=False)

print("=== Testing Model Inference ===")
print("\nRandom Sample Predictions:\n")

for i, idx in enumerate(random_indices, 1):
    sample = X_test_scaled[idx:idx+1]
    true_label = int(y_test[idx])
    
    # Make prediction
    prediction_probs = dl_model.predict(sample, verbose=0)[0]
    predicted_label = np.argmax(prediction_probs)
    confidence = prediction_probs[predicted_label] * 100
    
    status = "✓" if predicted_label == true_label else "✗"
    
    print(f"Sample {i}:")
    print(f"  True Activity: {activity_labels[true_label]}")
    print(f"  Predicted Activity: {activity_labels[predicted_label]}")
    print(f"  Confidence: {confidence:.2f}%")
    print(f"  Status: {status}")
    print()


## Summary

This notebook successfully trained a deep learning model for human activity recognition using the UCI HAR Dataset.

**Key Points:**
- Model architecture: Deep Neural Network with 4 hidden layers
- Input: 561 features from accelerometer and gyroscope data
- Output: 6 activity classes
- Target accuracy: >85%
- Model saved in multiple formats for deployment

**Next Steps:**
1. Use the saved models for API integration
2. Deploy TFLite model to mobile devices
3. Compare performance with Federated Learning model
